# Reliable DFU — status and evaluation (no training)
Reads the persistent Drive run, verifies completion, lists missing or resumable trials, and summarizes final metrics only when the required artifacts exist.

In [ ]:
import json, os, sys, subprocess
from pathlib import Path
import pandas as pd
from google.colab import drive

MOUNT=Path('/content/drive')
if not (MOUNT/'MyDrive').is_dir():
    drive.mount(str(MOUNT), force_remount=False)
RUN=MOUNT/'MyDrive'/'DFU-ImageGuard'/'runs'/'RELIABLE_DFU_CV_V1'
if not RUN.is_dir():
    raise FileNotFoundError(f'Run directory not found: {RUN}')

MODELS=('convnextv2_tiny','mobilenetv3_large','densenet121')
SEEDS=(2026,2027,2028)
FOLDS=(1,2,3,4,5)
expected=[(m,s,f) for f in FOLDS for s in SEEDS for m in MODELS]
completed=[]; missing=[]; resumable=[]; metric_rows=[]
for model,seed,fold in expected:
    trial=RUN/'trials'/model/f'seed_{seed}'/f'fold_{fold}'
    complete=trial/'COMPLETE.json'
    pred=trial/'test_predictions.csv'
    last=trial/'last_resume.pt'
    if complete.is_file() and pred.is_file():
        completed.append((model,seed,fold))
        try:
            metric_rows.append(json.loads(complete.read_text(encoding='utf-8')))
        except Exception as exc:
            metric_rows.append({'model_key':model,'seed':seed,'outer_fold':fold,'read_error':str(exc)})
    else:
        missing.append((model,seed,fold))
        if last.is_file():
            epoch='unknown'
            try:
                import torch
                payload=torch.load(last,map_location='cpu',weights_only=False)
                epoch=int(payload.get('epoch',-1))
            except Exception as exc:
                epoch=f'unreadable: {type(exc).__name__}'
            resumable.append({'model':model,'seed':seed,'fold':fold,'last_saved_epoch':epoch,'path':str(last)})

status={
    'run_root':str(RUN),
    'completed_trials':len(completed),
    'expected_trials':len(expected),
    'completion_percent':round(100*len(completed)/len(expected),2),
    'missing_trials':len(missing),
    'resumable_trials':resumable,
    'is_complete':len(completed)==len(expected),
}
(RUN/'RUN_STATUS_AUDIT.json').write_text(json.dumps(status,indent=2,default=str),encoding='utf-8')
print(json.dumps(status,indent=2,default=str))

if metric_rows:
    partial=pd.DataFrame(metric_rows)
    partial.to_csv(RUN/'tables'/'status_audit_available_trial_metrics.csv',index=False)
    display_cols=[c for c in ['model_key','seed','outer_fold','balanced_accuracy','sensitivity','specificity','roc_auc','pr_auc','brier','ece','fn','fp'] if c in partial.columns]
    print('\nAvailable completed-trial metrics:')
    display(partial[display_cols].sort_values([c for c in ['outer_fold','seed','model_key'] if c in display_cols]))

if not status['is_complete']:
    missing_df=pd.DataFrame(missing,columns=['model_key','seed','outer_fold'])
    missing_df.to_csv(RUN/'tables'/'missing_trials.csv',index=False)
    print('\nFINAL EVALUATION BLOCKED: run is incomplete.')
    print('The training notebook must be rerun with the storage-aware fixed commit. Completed trials will be skipped and resumable trials will continue.')
    display(missing_df.head(45))
else:
    required=[RUN/'tables'/'fold_seed_metrics.csv',RUN/'tables'/'all_oof_predictions.csv',RUN/'tables'/'model_summary.csv',RUN/'tables'/'selective_prediction.csv',RUN/'tables'/'error_audit.csv',RUN/'tables'/'paired_bootstrap.json',RUN/'FINAL_VERIFICATION.json']
    absent=[str(p) for p in required if not p.is_file()]
    if absent:
        print('All 45 trials are complete, but final report files are missing. Run the no-training regeneration notebook.')
        print(json.dumps({'missing_report_files':absent},indent=2))
    else:
        metrics=pd.read_csv(required[0])
        summary=pd.read_csv(required[2],header=[0,1],index_col=0)
        selective=pd.read_csv(required[3])
        errors=pd.read_csv(required[4])
        bootstrap=json.loads(required[5].read_text(encoding='utf-8'))
        final=json.loads(required[6].read_text(encoding='utf-8'))
        print('\nFINAL RUN VERIFIED: 45/45 trials complete')
        print(json.dumps(final,indent=2,default=str))
        print('\nModel summary:')
        display(summary)
        print('\nSelective prediction:')
        display(selective)
        print('\nError audit:')
        display(errors.head(50))
        print('\nPaired bootstrap:')
        print(json.dumps(bootstrap,indent=2,default=str))
